# 01 · 데이터 점검 & EDA

원본 확인 → 품질 점검 → Target·특성별 이탈률.
그래프마다: "[관찰] → 따라서 [처리/실험] → 다만 [인과/일반화] 단정 불가"

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, matplotlib.pyplot as plt
from src.predict import load_config, resolve_path

cfg = load_config()
df = pd.read_csv(resolve_path(cfg['data']['raw_path']), encoding=cfg['data']['encoding'], sep=cfg['data']['sep'])
tcol = cfg['target']['column']
pos = str(cfg['target']['positive_label']).strip().lower()
y = df[tcol].astype(str).str.strip().str.lower().eq(pos).astype(int)
print(df.shape, '| 중복', df.duplicated().sum())
df.head()

## 품질 요약 (dtype·결측·고유값)

In [ ]:
pd.DataFrame({'dtype': df.dtypes.astype(str), 'n_missing': df.isna().sum(),
              'pct_missing': (df.isna().mean()*100).round(2), 'n_unique': df.nunique()})

## Target 분포 & 특성별 이탈률

In [ ]:
y.value_counts(normalize=True).rename({0:'유지',1:'이탈'}).plot.bar(title='Churn rate'); plt.show()

cat_cols = [c for c in df.columns if c != tcol and not pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() <= 15]
for c in cat_cols[:4]:
    y.groupby(df[c]).mean().sort_values().plot.barh(title=f'{c} 별 이탈률'); plt.show()

## 정리 → reports/report.md, docs/project_plan.md
- 인사이트 3개 / 전처리·모델링 결정 3개 / 데이터 한계 1개
- 누수 의심·제거 컬럼, 제거할 ID 컬럼
- **확정한 정제 규칙은 src/clean.py 의 clean_raw() 에 코드로 옮기고 팀에 공지** (B·C가 import해서 동일 기준으로 시작 — handoff 카드 1번)